In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import time
import pandas as pd
from datetime import datetime
import requests
from copy import deepcopy

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'PT CMVM' ## change to current controller name
print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running PT CMVM Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict={

        regulatorName + ' 1': 'https://www.cmvm.pt/PInstitucional/Content?Input=EC2CC0691CC518A5DD27F600C912A72B95813803AA700D079B482F06F6A9D4D3',
        regulatorName + ' 2': 'https://www.cmvm.pt/PInstitucional/Content?Input=5DFE7211A0E7ECF9447CFDDEB7DF36130830865865A4241BD33893ED167B3991',
        regulatorName + ' 3': 'https://www.cmvm.pt/PInstitucional/Content?Input=DE69D31DE34B669FF11251BE9053B7BF8AF19CC5ECBEB434FC4E4A857452C478',
        regulatorName + ' 4': 'https://www.cmvm.pt/PInstitucional/Content?Input=E53C246B2452F00BEB3AEEF67C47C69CA0E3BB26DACB718CAD315564B7A421B4',
        regulatorName + ' 5': 'https://www.cmvm.pt/PInstitucional/Content?Input=185E6B62730853323BB4F2C2D727ACBA64584CC7E65E1DCDC5BDE00DA2BC7486',
        regulatorName + ' 6': 'https://www.cmvm.pt/PInstitucional/Content?Input=A569E0E41AC02102EA1881F785AF55733C331047139FDB06C96372769F55084F',
        regulatorName + ' 7': 'https://www.cmvm.pt/PInstitucional/Content?Input=6A7A54D1B9464E33BE3EBFA4FAA15FA2F45BE9F585803D7F3EFE1995C9FAD072'
        }



Typology={

       regulatorName + ' 1': 'Issuers',
       regulatorName + ' 2': 'Financial intermediaries registered with the CMVM',
       regulatorName + ' 3': 'Management Companies',
       regulatorName + ' 4': 'Investment funds',
       regulatorName + ' 5': 'Registered crowdfunding platform managers',
       regulatorName + ' 6': 'Financial intermediaries registered for providing investment advice',
       regulatorName + ' 7': 'Financial intermediaries that provide the services of investment research and financial analysis or other forms of general recommendation relating to transactions in financial instruments',

        }

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


In [ ]:



#------------------------------------------------ Begin_Main ----------------------------------------

# Shared HTTP session re-used by all API-based branches (L2 onwards).
_BASE_COOKIES = {
    "osVisit": "88283d6b-0f50-4eba-adbf-e0edd4d41562",
    "osVisitor": "1f90625d-94be-4d10-a8d2-b7a97a5454d6",
    "nr1Users": "lid%3dAnonymous%3btuu%3d0%3bexp%3d0%3brhs%3dXBC1ss1nOgYW1SmqUjSxLucVOAg%3d%3bhmc%3dEa1HzjBuUTncHx8CId%2bXGpoeEIQ%3d",
    "nr2Users": "crf%3dT6C%2b9iB49TLra4jEsMeSckDMNhQ%3d%3buid%3d0%3bunm%3d",
    "_pk_id.1.dc23": "d5e6c08ec4eafefb.1767022259.",
    "_pk_ses.1.dc23": "1",
}
def _fresh_session():
    s_ = requests.Session()
    s_.cookies.update(_BASE_COOKIES)
    return s_

for k, reg in enumerate(regdict):
    # print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    if reg == regulatorName + ' 1':
        # List 1: Issuers — 2 inner tabs, both backed by OutSystems endpoints.
        # Tab 1 = "Issuers of listed securities admitted to trading" (85 rows).
        # Tab 2 = "Other issuers / Public companies (existing feature until 31/12/2022)" (~430 rows).
        # Pulling MaxRecords=1000 in a single POST per tab is far more reliable than the
        # OutSystems Selenium tab/pagination dance.
        api_dict = {
            "EmitentesLst": {
                "url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Emitentes_CW/Emitentes/EmitentesLst/DataActionFetchEmitentes",
                "token": "8719422161413504",
                "apiVersion": "0xzZQR9g+_vETZcV18EuWA",
                "tab_label": "Issuers of listed securities admitted to trading",
            },
            "EmitentesOutrosLst": {
                "url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Emitentes_CW/Emitentes/EmitentesOutrosLst/DataActionFetchEmitentes_Outros",
                "token": "5036186261291820",
                "apiVersion": "L35ND2v40LSkZ+qkSAw+Kw",
                "tab_label": "Other issuers / Public companies (existing feature until 31/12/2022)",
            },
        }
        REFERER = regdict[reg]
        payload_common = {
            "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": ""},
            "viewName": "MainFlow.Content",
            "screenData": {"variables": {"StartIndex": 0, "MaxRecords": 1000, "Nome": "", "_nomeInDataFetchStatus": 1}},
        }
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        s = _fresh_session()

        total = 0
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}
            payload = deepcopy(payload_common)
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            r = s.post(api["url"], json=payload, headers=headers, timeout=30)
            # print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            data = r.json().get("data", {})

            # Find the rows list (key name differs per tab: EmitentesLst3, EmitentesOutrosLst, ...).
            rows = []
            for v in data.values():
                if isinstance(v, dict) and "List" in v:
                    rows = v["List"]
                    break
            # print(f"  {api_name}: {len(rows)} entities (server TotalCount={data.get('TotalCount')})")
            total += len(rows)

            for li in rows:
                sqldict['Name'].append(li.get('NOM_ENT'))
                sqldict['InternalID_1'].append(li.get('NUM_ENT'))
                sqldict['InternalID_1_type'].append('Entity number')
                sqldict['Typology'].append(api["tab_label"])
                sqldict['ListName'].append(Typology[reg])
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['Cntry'].append('PT')
                sqldict = bourange_same_length_array(sqldict)

        # print(f"List 1: collected {total} entities across {len(api_dict)} tabs.")


    elif reg == regulatorName + ' 2':
        # List 2: Financial intermediaries registered with the CMVM.
        # Selenium accordion logic was unreliable (hardcoded section titles never matched);
        # use the OutSystems API directly. Returns ~87 entities.
        POST_URL_LIST = 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_IntermediariosFin_CW/Main/IntermediariosfinanceirosListWB/DataActionGet_Data'
        POST_URL = 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/IntermediarioFinanceirosDetail/DataActionGetIntermediarioFinanceiroByNum_ENT'
        REFERER = regdict[reg]

        list_payload = {
            "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "FZtiBLy61eUFrZmiLRYS+g"},
            "viewName": "MainFlow.Content",
            "screenData": {"variables": {
                "StartIndex": 0, "MaxRecord": 1000,
                "Search_Name": '', "_search_NameInDataFetchStatus": 1,
                "Tipo": '', "_tipoInDataFetchStatus": 1,
                "ServicoRowNumber": 0, "_servicoRowNumberInDataFetchStatus": 1,
            }},
        }
        payload = {"versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "YoH8QlSvlZ7V95iJQDhtJg"},
                   "viewName": "MainFlow.Content",
                   "screenData": {"variables": {"NUM_ENT": '', "_nUM_ENTInDataFetchStatus": 1}}}

        s = _fresh_session()
        headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "outsystems-request-token": "7701502406934197",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }

        r = s.post(POST_URL_LIST, json=list_payload, headers=headers, timeout=30)
        r.raise_for_status()
        nom_ent_list = r.json()['data']['DataStructureList']['List']
        # print(f"List 2: fetched {len(nom_ent_list)} entities")

        for li in nom_ent_list:
            payload['screenData']['variables']['NUM_ENT'] = li['ENT_NUM_ENT']
            r = s.post(POST_URL, json=payload, headers=headers, timeout=30)
            r.raise_for_status()
            d = r.json()['data']['IntermediarioFinanceiro']

            sqldict['Name'].append(d.get('nom_ent') or li.get('NOM_ENT'))
            sqldict['Address_1'].append(d.get('mor_ent'))
            sqldict['Zip'].append(d.get('gr_cod_pos'))
            sqldict['City'].append(d.get('fr_cod_dsc'))
            sqldict['RegulationDate'].append(d.get('inicio_act'))
            sqldict['InternalID_1'].append(d.get('num_ctb'))
            sqldict['InternalID_1_type'].append('Tax identification number')
            sqldict['Typology'].append(d.get('tipo'))
            sqldict['ListName'].append(Typology[reg])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict["Cntry"].append("PT")
            sqldict = bourange_same_length_array(sqldict)

    elif reg == regulatorName + ' 3':

        api_dict = {
            "SociedadesGestoras": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetchSociedadesGestoras",
                                "token": "3706222453791299",
                                    "apiVersion": "PgGljBGtlHspvDFlgMm1Fw"
                                },
            "Empresas": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_EmpresasSeguros",
                        "token": "7701502406934197",
                        "apiVersion": "iAWZxkjpVvfaTAY9WAMoCg"},
            "STC": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_STC",
                    "token": "2616747537848431",
                    "apiVersion": "OJsVJq3Gu366Y7XC1R5eVA"},
            "EGFP": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_EGFP",
                    "token": "4027971824164562",
                    "apiVersion": "z9PkhiYn1+MuaB4nfPY0bg"},
            "SGFTC": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_SGFTC",
                    "token": "8040918438076384",
                    "apiVersion": "OFPYhNHcRG6a_g0p2sZigA"},
        }

        REFERER = regdict[reg]

        payload_common = {"versionInfo":{"moduleVersion":"ctTAXtPi7v5Pjymrl1lceA","apiVersion":"PgGljBGtlHspvDFlgMm1Fw"},
                            "viewName":"MainFlow.Content",
                            "screenData":{"variables":{"StartIndex_SociedadesGestoras":0,"MaxRecords":1000,"StartIndex_STC":0,"StartIndex_SGFTC":0,"StartIndex_EmpresasSeguros":0,"StartIndex_EGFP":0,"IsEmpty_SociedadesGestoras_1":'false',"IsEmpty_SGFTC_2":'false',"IsEmpty_STC_3":'false',"IsEmpty_ES_4":'false',"IsEmpty_EGFP_5":'false',

                                                    "Entity_Name":"","_entity_NameInDataFetchStatus":1,"IsActive":'true',"_isActiveInDataFetchStatus":1,"Society_Type":"","_society_TypeInDataFetchStatus":1,"Dimension":"","_dimensionInDataFetchStatus":1,"Ambito":"","_ambitoInDataFetchStatus":1}}}


        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }

        start_index_key = {
            "SociedadesGestoras": "StartIndex_SociedadesGestoras",
            "Empresas": "StartIndex_EmpresasSeguros",
            "STC": "StartIndex_STC",
            "EGFP": "StartIndex_EGFP",
            "SGFTC": "StartIndex_SGFTC",
        }

        s = _fresh_session()
        results = {}

        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}
            payload = deepcopy(payload_common)
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            payload["screenData"]["variables"][start_index_key[api_name]] = 0

            r = s.post(api["url"], json=payload, headers=headers, timeout=30)
            # print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()

        infors = []
        for result in results:
            # print(result)
            data = results[result].get('data', {})
            if 'List' in data:
                infors.append(data['List'])
                continue
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
            if result == "SociedadesGestoras" or result == "STC":
                if result == "SociedadesGestoras":
                    iterate_list =  infors[0]
                else:
                    iterate_list =  infors[2]
                for li in iterate_list:
                    update_code = li['NUM_ENT']
                    # print(update_code)
                    if result == "SociedadesGestoras":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_CapRiscoEmp_CW/UIFlow1/CapRiscoEmp_SociedadeDetail_Wb/DataActionGetSociedadeDetail"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "hk5mA_EktvXi0UTawTX_sQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "Num_Ent": str(li["NUM_ENT"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    elif result == "STC":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/StcDetail/DataActionGetSct"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "52l_VyMgXpGRY3naCd3aWg"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_ENT": str(li["NUM_ENT"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                    r.raise_for_status()
                    detail_data = r.json()
                    if result == "SociedadesGestoras":
                        name_ = li['NOM_ENT']
                        address_1 = detail_data['data']['SociedadeDetails']['mor_ent']
                        zip_code_ = detail_data['data']['SociedadeDetails']['gr_cod_pos']
                        city_ = detail_data['data']['SociedadeDetails']['gr_cod_dsc']
                        register_date = detail_data['data']['SociedadeDetails']['data_reg']
                        tax_id = detail_data['data']['SociedadeDetails']['num_ctb']
                        tipo = detail_data['data']['SociedadeDetails']['tipo']
                        # print(name_, address_1, zip_code_, city_, register_date, tax_id, tipo)
                    elif result == "STC":
                        name_ = li['NOM_ENT']
                        address_1 = detail_data['data']['Sct']['mor_stc']
                        zip_code_ = detail_data['data']['Sct']['cod_pst']
                        city_ = detail_data['data']['Sct']['gr_cod_dsc_abr']
                        register_date = detail_data['data']['Sct']['dat_reg']
                        register_number = detail_data['data']['Sct']['num_reg']
                        tax_id = detail_data['data']['Sct']['num_ctb']
                        tipo = detail_data['data']['Sct']['tipo']
                        # print(name_, address_1, zip_code_, city_, register_date, tax_id, tipo)

                    sqldict['Name'].append(name_)
                    sqldict['Address_1'].append(address_1)
                    sqldict['Zip'].append(zip_code_)
                    sqldict['City'].append(city_)
                    sqldict['RegulationDate'].append(register_date)
                    sqldict['InternalID_1'].append(tax_id)
                    sqldict['InternalID_1_type'].append('Tax identification number')
                    sqldict['Typology'].append(tipo)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['Cntry'].append('PT')
                    sqldict = bourange_same_length_array(sqldict)
            elif result == "Empresas" or result == "EGFP" :
                if result == "Empresas":
                    iterate_list =  infors[1]
                elif result == "EGFP":
                    iterate_list =  infors[3]


                for li in iterate_list:
                    update_code = li['Value']
                    if result == "Empresas":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/EmpresaSeguroDetail/DataActionGetData"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "eGyCcqSS6uhvu9nqqGAMUQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_ENT": str(li["Value"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    elif result == "EGFP":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/EmpresaSeguroDetail/DataActionGetData"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "eGyCcqSS6uhvu9nqqGAMUQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_ENT": str(li["Value"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                    r.raise_for_status()
                    detail_data = r.json()
                    payload_block = detail_data['data'].get('EmpresaSeguro') or {}
                    name_ = payload_block.get('Nom_ent') or li.get('Text')
                    address_1 = payload_block.get('mor_ent')
                    zip_code_ = payload_block.get('Cod_pst')
                    city_ = payload_block.get('gr_cod_dsc')
                    email_ = payload_block.get('email')
                    website_ = payload_block.get('site_ent')
                    if result == "EGFP" and not payload_block.get('Nom_ent'):
                        print(f"[WARN] EGFP {update_code}: empty EmpresaSeguro payload — using list-page name")

                    sqldict['Name'].append(name_)
                    sqldict['Address_1'].append(address_1)
                    sqldict['Zip'].append(zip_code_)
                    sqldict['City'].append(city_)
                    sqldict['Email'].append(email_)
                    sqldict['Website'].append(website_)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict["Cntry"].append("PT")
                    sqldict = bourange_same_length_array(sqldict)
            elif result == "SGFTC":
                iterate_list =  infors[4]
                for li in iterate_list:
                    name_ = li['Text']
                    # print(name_)
                    sqldict['Name'].append(name_)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict["Cntry"].append("PT")
                    sqldict = bourange_same_length_array(sqldict)

    elif reg == regulatorName + ' 4':
        api_dict = {
            "Pensoes": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosPensoes",
                                "token": "7011207507672761",
                                "apiVersion": "_Pgt+oS9kB0YCiQR5L8OBA"
                                },
            "RecuperaCreditos": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosRecuperaCreditos",
                        "token": "6119502279184928",
                        "apiVersion": "o_5_n+NaxoawcbIxlzMN7Q"},
            "Mobiliario": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosInvestimentoMobiliario",
                    "token": "6832074922633300",
                    "apiVersion": "2Kyh7s0bYwCUR_MKDQ02nQ"},
            "Investimento": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchSociedadesInvestimento",
                    "token": "1048115306215631",
                    "apiVersion": "TUTaqt2hCTUkF8T98ecIdQ"},
            "CapitalRisco": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosCapitalRisco",
                    "token": "8606747129710045",
                    "apiVersion": "aMaXSzQtee_WdMh_bxtgew"},
            "FundosInvestimentoIMobiliario": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosInvestimentoIMobiliario",
                    "token": "8665773788672252",
                    "apiVersion": "lHjTvAr3axK4WTTLmOMdwQ"},
            "FundosTitularizacaoCredito": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchTitularizacaoCredito",
                    "token": "2730594537642582",
                    "apiVersion": "A7L1T3Y6Y4sxiOT1Qts+1Q"},
            "SocialeAlternativo": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundoEmpreendedorismoSocialeAlternativo",
                "token": "2526903115018427",
                "apiVersion": "xOdD+tVclDPcmX+gYQcDPQ"},
        }

        REFERER = regdict[reg]

        payload_common = {
        "versionInfo": {"moduleVersion":"ctTAXtPi7v5Pjymrl1lceA","apiVersion":"xOdD+tVclDPcmX+gYQcDPQ"},
        "viewName":"MainFlow.Content",
        "screenData":{"variables":{
            "MaxRecords":1000,
            "StartIndex_FII":0,
            "StartIndex_FIM":0,
            "StartIndex_FCR":0,
            "StartIndex_SIC":0,
            "StartIndex_FTC":0,
            "StartIndex_FRC":0,
            "StartIndex_FP":0,
            "StartIndex_FC":0,
            "StartIndex_FES":0,
            "StartIndex_SES":0,
            "Input_IsActive": True,
            "_input_IsActiveInDataFetchStatus": 1,
            "Input_Num_Fundo": "",
            "_input_Num_FundoInDataFetchStatus": 1,
            "Input_Nom_SubFundo": "",
            "_input_Nom_SubFundoInDataFetchStatus": 1,
            "Input_Num_EntGestora": "",
            "_input_Num_EntGestoraInDataFetchStatus": 1
        }}
        }

        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }

        s = _fresh_session()
        results = {}

        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}
            payload = deepcopy(payload_common)
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            r = s.post(api["url"], json=payload, headers=headers, timeout=30)
            # print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()

        infors = []
        for result in results:
            # print(result)
            data = results[result].get('data', {})
            if 'List' in data:
                infors.append(data['List'])
                continue
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")

        for k_, v in results.items():
            if k_ == "Pensoes":
                iterate_list =  infors[0]
                POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/Fundo_Pensao_Detail/DataActionGetFundo"
                for li in iterate_list:
                    update_code = li['num_prd']
                    # print(update_code)
                    payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "cnDvjEOyf_gWI8s63cVoOA"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_FUN": str(li["num_prd"]),
                                "_nUM_FUNInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                    r.raise_for_status()
                    detail_data = r.json()
                    name_ = detail_data['data']['Fundo']['nom_prd']
                    management_company_ = detail_data['data']['Fundo']['nom_ent']
                    tipo_ = detail_data['data']['Fundo']['desc_fun']
                    sqldict['Name'].append(name_)
                    sqldict['Name - Mother Company'].append(management_company_)
                    sqldict['Typology'].append(tipo_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['Cntry'].append('PT')
                    sqldict = bourange_same_length_array(sqldict)

            elif k_ in ("Mobiliario", "FundosInvestimentoIMobiliario", "RecuperaCreditos", "Investimento", "CapitalRisco", "SocialeAlternativo"):
                if k_ == "Mobiliario":
                    iterate_list =  infors[2]
                elif k_ == "RecuperaCreditos":
                    iterate_list =  infors[1]
                elif k_ == "Investimento":
                    iterate_list =  infors[3]
                elif k_ == "CapitalRisco":
                    iterate_list =  infors[4]
                elif k_ == 'FundosInvestimentoIMobiliario':
                    iterate_list =  infors[5]
                elif k_ == "SocialeAlternativo":
                    iterate_list =  infors[7]
                POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_CapRiscoEmp_CW/UIFlow1/CapRiscoEmp_FundoDetail_Wb/DataActionGetFundoDetail"
                for li in iterate_list:
                    update_code = li['NUM_FUN']
                    payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "j_bMr8ysi3iQPkqYVOBZUw"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "Fun_Num": str(li["NUM_FUN"]),
                                "_fun_NumInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                    r.raise_for_status()
                    detail_data = r.json()
                    name_ = detail_data['data']['Out_FundoDetail']['Fundo_SimpleStructure']['descom_fun']
                    management_company_ = detail_data['data']['Out_FundoDetail']['Fundo_SimpleStructure']['ent_gestora']
                    isin_code_ = detail_data['data']['Out_FundoDetail']['Fundo_SimpleStructure']['cod_isi']
                    register_date_ = detail_data['data']['Out_FundoDetail']['Fundo_SimpleStructure']['data_inicio']
                    fund_code = detail_data['data']['Out_FundoDetail']['Fundo_SimpleStructure']['num_fun']
                    tipo_ = detail_data['data']['Out_FundoDetail']['Fundo_SimpleStructure']['tipo']
                    sqldict['Name'].append(name_)
                    sqldict['Name - Mother Company'].append(management_company_)
                    sqldict['InternalID_1'].append(isin_code_)
                    sqldict['InternalID_1_type'].append('ISIN code')
                    sqldict['RegulationDate'].append(register_date_)
                    sqldict['InternalID_2'].append(fund_code)
                    sqldict['InternalID_2_type'].append('Fund code')
                    sqldict['Typology'].append(tipo_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['Cntry'].append('PT')
                    sqldict = bourange_same_length_array(sqldict)

            elif k_ == "FundosTitularizacaoCredito":
                iterate_list =  infors[6]
                POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/FundoDetail/DataActionGetFundo"
                for li in iterate_list:
                    update_code = li['Id']
                    payload_detail = {
                        "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "nhcP2ss2TGTf_aMr0ZpBCQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_FUN": str(li["Id"]),
                                "_nUM_FUNInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                    r.raise_for_status()
                    detail_data = r.json()
                    name_ = detail_data['data']['Fundo']['FundoDetail_Simple']['nome']
                    management_company_ = detail_data['data']['Fundo']['FundoDetail_Simple']['gestora']
                    isin_code_ = detail_data['data']['Fundo']['FundoDetail_Simple']['cod_isi']
                    register_date_ = detail_data['data']['Fundo']['FundoDetail_Simple']['data_inicio']
                    tipo_ = detail_data['data']['Fundo']['FundoDetail_Simple']['tipo']
                    sqldict['Name'].append(name_)
                    sqldict['Name - Mother Company'].append(management_company_)
                    sqldict['InternalID_1'].append(isin_code_)
                    sqldict['InternalID_1_type'].append('ISIN code')
                    sqldict['RegulationDate'].append(register_date_)
                    sqldict['Typology'].append(tipo_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['Cntry'].append('PT')
                    sqldict = bourange_same_length_array(sqldict)

    elif reg == regulatorName + ' 5':
        api_dict = {
            "Crowdfunding": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_FinColaborativo_CW/UIFlow1/RegisteredEntityCrowdfunding/DataActionGetRegisteredEntityCrowdfunding",
                                "token": "1627131176195664",
                                    "apiVersion": "E+dt2ieyrhwZ3OGir57dxQ"
                                }}
        payload_common = {"versionInfo":{"moduleVersion":"ctTAXtPi7v5Pjymrl1lceA","apiVersion":"PgGljBGtlHspvDFlgMm1Fw"},
                            "viewName":"MainFlow.Content",
                            "screenData":{"variables":{"StartIndex":0,"MaxRecords":1000,}}}
        REFERER = regdict[reg]
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        s = _fresh_session()
        results = {}
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}
            payload = deepcopy(payload_common)
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            r = s.post(api["url"], json=payload, headers=headers, timeout=30)
            # print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()

        infors = []
        for result in results:
            # print(result)
            data = results[result].get('data', {})
            if 'List' in data:
                infors.append(data['List'])
                continue
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
        iterate_list =  infors[0]
        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_FinColaborativo_CW/UIFlow1/RegisteredEntityCrowdfunding_Detail/DataActionGetRegisteredCrowdfundingPlatform"
        for li in iterate_list:
                update_code = li['num_ent']
                # print(update_code)
                payload_detail = {
                    "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "DjQJaplTYz_oV+8CsmnMVA"},
                    "viewName": "MainFlow.Content",
                    "screenData": {
                        "variables": {
                            "NUM_ENT": str(li["num_ent"]),
                            "_nUM_ENTInDataFetchStatus": 1,
                        }
                    },
                }
                r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                r.raise_for_status()
                detail_data = r.json()
                name_ = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['NOM_ENT']
                tip_fin = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['TIP_FIN']
                mor_ent = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['MOR_ENT']
                cod_pst = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['COD_PST']
                gr_cod_dsc = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['GR_COD_DSC']
                dat_reg = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['DAT_REG']
                sqldict['Name'].append(name_)
                sqldict['Typology'].append(tip_fin)
                sqldict['Address_1'].append(mor_ent)
                sqldict['Zip'].append(cod_pst)
                sqldict['City'].append(gr_cod_dsc)
                sqldict['RegulationDate'].append(dat_reg)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict['Cntry'].append('PT')
                sqldict = bourange_same_length_array(sqldict)

    elif reg == regulatorName + ' 6':
        api_dict = {
            "Financial intermediaries registered for providing investment advices": {"url": 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/FinancialIntermediaryList_CW/DataActionFetchFinancialIntermediaryList',
                                    "token": "2280863588950865",
                                    "apiVersion": "nEjGGjihelof6xGGKHyoaw"
                                }}
        payload_common = {"versionInfo":{"moduleVersion":"ctTAXtPi7v5Pjymrl1lceA","apiVersion":"nEjGGjihelof6xGGKHyoaw"},"viewName":"MainFlow.Content","screenData":{"variables":{"StartIndex":0,"MaxRecord":1000,"ServNum":5,"_servNumInDataFetchStatus":1,"Search_EntityName":"","_search_EntityNameInDataFetchStatus":1,"Search_Type":"","_search_TypeInDataFetchStatus":1}}}
        REFERER = regdict[reg]
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        s = _fresh_session()
        results = {}
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}
            payload = deepcopy(payload_common)
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            r = s.post(api["url"], json=payload, headers=headers, timeout=30)
            r.raise_for_status()
            results[api_name] = r.json()
        infors = []
        for result in results:
            data = results[result].get('data', {})
            if 'List' in data:
                infors.append(data['List'])
                continue
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
        iterate_list =  infors[0]
        POST_URL_DETAIL =  'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/IntermediarioFinanceirosDetail/DataActionGetIntermediarioFinanceiroByNum_ENT'
        for li in iterate_list:
                update_code = li['NUM_ENT']
                payload_detail = {
                    "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "YoH8QlSvlZ7V95iJQDhtJg"},
                    "viewName": "MainFlow.Content",
                    "screenData": {
                        "variables": {
                            "NUM_ENT": str(li["NUM_ENT"]),
                            "_nUM_ENTInDataFetchStatus": 1,
                        }
                    },
                }
                r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                r.raise_for_status()
                detail_data = r.json()
                d = detail_data['data']['IntermediarioFinanceiro']
                name_ = d.get('nom_ent')
                address_1 = d.get('mor_ent')
                zip_code_ = d.get('gr_cod_pos')
                city_ = d.get('fr_cod_dsc')
                tipo = d.get('tipo')
                email_ = d.get('email')
                tax_id = d.get('num_ctb')
                inner_num = d.get('num_reg')

                sqldict['Name'].append(name_)
                sqldict['Typology'].append(tipo)
                sqldict['Address_1'].append(address_1)
                sqldict['Zip'].append(zip_code_)
                sqldict['City'].append(city_)
                sqldict['Email'].append(email_)
                sqldict['InternalID_1'].append(tax_id)
                sqldict['InternalID_1_type'].append('Tax identification number')
                sqldict['InternalID_2'].append(inner_num)
                sqldict['InternalID_2_type'].append('Registration number with the CMVM')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict['Cntry'].append('PT')
                sqldict = bourange_same_length_array(sqldict)

    elif reg == regulatorName + ' 7':
        api_dict = {
            "Financial intermediaries providing investment research/analysis": {"url": 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_AnalFinanceiros_CW/Main/IntermediariosFinanceirosList_Wb/DataActionGetIntermediariosFinanceiros',
                                    "token": "6322880696288143",
                                    "apiVersion": "YLLqP7MQZUzxA7E74_xhMA"
                                }}
        payload_common = {"versionInfo":{"moduleVersion":"ctTAXtPi7v5Pjymrl1lceA","apiVersion":"YLLqP7MQZUzxA7E74_xhMA"},"viewName":"MainFlow.Content","screenData":{"variables":{"StartIndex":0,"MaxRecords":1000,"Search_EntityName":"","_search_EntityNameInDataFetchStatus":1,"Search_Type":"","_search_TypeInDataFetchStatus":1}}}
        REFERER = regdict[reg]
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        s = _fresh_session()
        results = {}
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}
            payload = deepcopy(payload_common)
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            r = s.post(api["url"], json=payload, headers=headers, timeout=30)
            r.raise_for_status()
            results[api_name] = r.json()
        infors = []
        for result in results:
            # print(result)
            data = results[result].get('data', {})
            if 'List' in data:
                infors.append(data['List'])
                continue
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
        iterate_list =  infors[0]
        POST_URL_DETAIL =  'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/IntermediarioFinanceirosDetail/DataActionGetIntermediarioFinanceiroByNum_ENT'
        for li in iterate_list:
                update_code = li['Entity_Number']
                payload_detail = {
                    "versionInfo": {"moduleVersion": "ctTAXtPi7v5Pjymrl1lceA", "apiVersion": "YoH8QlSvlZ7V95iJQDhtJg"},
                    "viewName": "MainFlow.Content",
                    "screenData": {
                        "variables": {
                            "NUM_ENT": str(li["Entity_Number"]),
                            "_nUM_ENTInDataFetchStatus": 1,
                        }
                    },
                }
                r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers, timeout=30)
                r.raise_for_status()
                detail_data = r.json()
                d = detail_data['data']['IntermediarioFinanceiro']
                name_ = d.get('nom_ent') or li.get('Entity_Name')
                if not d.get('nom_ent'):
                    print(f"[WARN] L7 NUM_ENT={update_code}: empty IntermediarioFinanceiro — using list-page name")
                address_1 = d.get('mor_ent')
                zip_code_ = d.get('gr_cod_pos')
                city_ = d.get('fr_cod_dsc')
                tipo = d.get('tipo') or li.get('Descricao')
                email_ = d.get('email')
                tax_id = d.get('num_ctb')

                sqldict['Name'].append(name_)
                sqldict['Typology'].append(tipo)
                sqldict['Address_1'].append(address_1)
                sqldict['Zip'].append(zip_code_)
                sqldict['City'].append(city_)
                sqldict['Email'].append(email_)
                sqldict['InternalID_1'].append(tax_id)
                sqldict['InternalID_1_type'].append('Tax identification number')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict['Cntry'].append('PT')
                sqldict = bourange_same_length_array(sqldict)


[INFO] : Working 1/7 _(PT CMVM 1)_ 
EmitentesLst 200 {"versionInfo":{"hasModuleVersionChanged":false,"hasApiVersionChanged":false},"data":{"EmitentesLst3":{"List":[{"NOM_ENT":"ALTRI, S.G.P.S., S.A.","NUM_ENT":"51988"},{"NOM_ENT":"Ares Lusitani - STC, S.
  EmitentesLst: 85 entities (server TotalCount=85)
EmitentesOutrosLst 200 {"versionInfo":{"hasModuleVersionChanged":false,"hasApiVersionChanged":false},"data":{"EmitentesOutrosLst":{"List":[{"NOM_ENT":"A. Silva & Silva - Imobiliário e Serviços, SA","NUM_ENT":"103698"},{"NOM
  EmitentesOutrosLst: 430 entities (server TotalCount=430)
List 1: collected 515 entities across 2 tabs.
[INFO] : Working 2/7 _(PT CMVM 2)_ 
List 2: fetched 87 entities
[INFO] : Working 3/7 _(PT CMVM 3)_ 
SociedadesGestoras 200 {"versionInfo":{"hasModuleVersionChanged":false,"hasApiVersionChanged":false},"data":{"SociedadesGestoras":{"List":[{"NUM_ENT":"180696","NOM_ENT":"Above Capital - SCR, S.A.","TIP_REL":"SCRP","OICVM":"
Empresas 200 {"versionInfo":{"hasModuleVer

In [6]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
sqldict = bourange_same_length_array(sqldict)   
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

# driver.quit()


In [7]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 1987 values.
Key 'priority' has 1987 values.
Key 'ListLabel' has 1987 values.
Key 'Typology' has 1987 values.
Key 'EntryType' has 1987 values.
Key 'Name' has 1987 values.
Key 'InternalID_1' has 1987 values.
Key 'InternalID_1_type' has 1987 values.
Key 'InternalID_2' has 1987 values.
Key 'InternalID_2_type' has 1987 values.
Key 'InternalID_3' has 1987 values.
Key 'InternalID_3_type' has 1987 values.
Key 'CoType' has 1987 values.
Key 'License_Type' has 1987 values.
Key 'Address_1' has 1987 values.
Key 'Address_2' has 1987 values.
Key 'City' has 1987 values.
Key 'Zip' has 1987 values.
Key 'Cntry' has 1987 values.
Key 'Phone' has 1987 values.
Key 'Fax' has 1987 values.
Key 'Website' has 1987 values.
Key 'Email' has 1987 values.
Key 'RegulationType' has 1987 values.
Key 'RegulationTypeCode' has 1987 values.
Key 'RegulationDate' has 1987 values.
Key 'CancellationDate' has 1987 values.
Key 'RegCtry' has 1987 values.
Key 'RegCode' has 1987 values.
Key 'ListCode' has 1987 values